# 04 — The feasible set, and why terrain inverts the climate-only answer

`F(dT) = C(dT) INTERSECT S`. `C` is the thermal band, which migrates as the climate warms. `S` is the terrain footprint coffee actually occupies, which does not move.

This notebook contains the paper's central result: screening a climate-suitability projection by feasible ground **reverses its sign**, and the feasible set is at a maximum at approximately present climate, so every pathway is downhill.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, json
import pipeline as P

isl = P.load_island(); FT, FTemp, reg, xy = P.load_farms(isl)
idx = np.arange(len(FT))
s = P.screen(FT, reg, isl['X'], idx)
print(f"screen threshold {s['threshold']:.3f}")
print(f"footprint: kona-like {s['kona_like'].sum():,} cells, "
      f"kau-like {s['kau_like'].sum():,} of {len(isl['X']):,} land cells")

screen threshold 4.071
footprint: kona-like 2,549 cells, kau-like 3,593 of 42,524 land cells


## The inversion

Climate-only, warming moves a large fraction of the island *into* the thermal band. Restricted to ground coffee can actually occupy, the same warming moves most of the belt *out* of it. The ratio between those two answers is the headline.

In [2]:
DT = 1.35
rows = []
for r, like in (('kona', s['kona_like']), ('kau', s['kau_like'])):
    mu, sg = P.envelope(FTemp, reg, idx, r)
    farm = isl['farmable']
    # CLIMATE ONLY: a cell "gains" if warming moves it CLOSER to the optimum,
    # i.e. T < mu - dT/2. This is the paper's definition (registered as
    # islandgain_*), not "newly enters the level set" -- the latter is a much
    # stricter set and gives ~11%, which is a different quantity.
    gain_iso = mu - DT/2
    gain_c = 100 * (farm & (isl['T'] < gain_iso)).sum() / farm.sum()
    # SAME QUESTION, screened by terrain the crop can actually occupy
    band0 = np.abs(isl['T'] - mu) <= P.HW*sg
    band1 = np.abs(isl['T'] + DT - mu) <= P.HW*sg
    f0 = (band0 & like & farm).sum(); f1 = (band1 & like & farm).sum()
    # residual gaining ground as a share of ALL farmable island land
    resid = 100 * (farm & like & band1 & ~band0).sum() / farm.sum()
    rows.append([r, gain_iso, gain_c, resid, gain_c/resid, 100*(f1-f0)/f0])
t = pd.DataFrame(rows, columns=['district','gain isotherm (C)','climate-only gain %',
                                'residual gain %','overestimate x','|F| change %'])
print(t.round(2).to_string(index=False))

pn = json.load(open('../paper_numbers.json'))
print(f"\nregression check: climate-only gain kona {rows[0][2]:.1f}% vs registered "
      f"{pn['islandgain_kona_2045']:.1f}%   isotherm {rows[0][1]:.2f} vs {pn['gainT_kona_2045']:.2f}")
assert abs(rows[0][2] - pn['islandgain_kona_2045']) < 1.0
print('matches')
print('\nTHE INVERSION: climate alone says ~70% of farmable island land gains')
print('thermal suitability. Restrict to ground coffee can occupy and the feasible')
print('set SHRINKS, with only a small residual gaining -- the overestimate factor.')

district  gain isotherm (C)  climate-only gain %  residual gain %  overestimate x  |F| change %
    kona              20.32                69.82             1.14           61.40        -26.41
     kau              20.15                68.45             1.78           38.38        -18.19

regression check: climate-only gain kona 69.8% vs registered 69.8%   isotherm 20.32 vs 20.32
matches

THE INVERSION: climate alone says ~70% of farmable island land gains
thermal suitability. Restrict to ground coffee can occupy and the feasible
set SHRINKS, with only a small residual gaining -- the overestimate factor.


## The matched comparison — screen off vs screen on

The earlier cell compares two *different* statistics: a directional gradient measure
over all farmable land ("moves closer to the optimum") against a set-cardinality
change over the footprint. A cell can move closer to the optimum and still sit
outside `C`, so that pairing cannot establish that `S` causes the sign flip.

The matched statistic is the same quantity computed twice: `|C(dT)|/|C(0)|` with the
screen off, and `|C(dT) cap S|/|C(0) cap S|` with it on.

In [3]:
# MATCHED: identical statistic, screen off vs screen on
matched = {}
print(f"{'unit':>8} {'dT':>6} {'|C|':>8} {'dC%':>7} {'|F|':>8} {'dF%':>7}")
for r, like in (('kona', s['kona_like']), ('kau', s['kau_like'])):
    mu, sg = P.envelope(FTemp, reg, idx, r)
    c0 = P.feasible_size(isl['T'], isl['farmable'], mu, sg, 0.0)
    f0 = P.feasible_size(isl['T'], isl['farmable'] & like, mu, sg, 0.0)
    for h, dt in P.DT_HORIZON.items():
        c = P.feasible_size(isl['T'], isl['farmable'], mu, sg, dt)
        f = P.feasible_size(isl['T'], isl['farmable'] & like, mu, sg, dt)
        matched[f'C_change_{r}_{h}'] = 100*(c-c0)/c0
        matched[f'F_change_{r}_{h}'] = 100*(f-f0)/f0
        print(f'{r:>8} {dt:+6.2f} {c:8,d} {100*(c-c0)/c0:+7.1f} {f:8,d} {100*(f-f0)/f0:+7.1f}')

# pooled: union of the two bands (screen off) against the pooled union (screen on)
def pooled_C(dt):
    u = np.zeros(len(isl['X']), bool)
    for r in ('kona', 'kau'):
        mu, sg = P.envelope(FTemp, reg, idx, r)
        u |= isl['farmable'] & (np.abs(isl['T'] + dt - mu) <= P.HW * sg)
    return int(u.sum())

c0 = pooled_C(0.0); f0, _ = P.pooled_union(isl, FT, FTemp, reg, dt=0.0)
for h, dt in P.DT_HORIZON.items():
    c = pooled_C(dt); f, _ = P.pooled_union(isl, FT, FTemp, reg, dt=dt)
    matched[f'C_change_pooled_{h}'] = 100*(c-c0)/c0
    matched[f'F_change_pooled_{h}'] = 100*(f-f0)/f0
    print(f'{"pooled":>8} {dt:+6.2f} {c:8,d} {100*(c-c0)/c0:+7.1f} {f:8,d} {100*(f-f0)/f0:+7.1f}')

# S's share of farmable land -- the reason the old "overestimate factor" was mostly
# mechanical: 69.8 / 1.1 is close to the reciprocal of this.
u_any = s['kona_like'] | s['kau_like']
matched['S_share_farmable_pct'] = 100*(isl['farmable'] & u_any).sum()/isl['farmable'].sum()
print(f"\nS covers {matched['S_share_farmable_pct']:.1f}% of farmable land")
print('\nThe thermal set alone is FLAT. The contraction is attributable to S,')
print('which is what the unmatched 69.8%-vs-22.1% pairing could not establish.')


    unit     dT      |C|     dC%      |F|     dF%
    kona  +1.00    9,417    +0.3    1,349   -18.3
    kona  +1.35    9,365    -0.3    1,215   -26.4
     kau  +1.00    7,134    +1.7    1,660    -9.6
     kau  +1.35    7,174    +2.2    1,502   -18.2
  pooled  +1.00    9,417    +0.3    3,009   -13.7
  pooled  +1.35    9,365    -0.3    2,717   -22.1

S covers 15.2% of farmable land

The thermal set alone is FLAT. The contraction is attributable to S,
which is what the unmatched 69.8%-vs-22.1% pairing could not establish.


## When does the screen flip the sign? The condition, and where it comes from

`|C|` and `|F|` change per unit `dT` by the same rule — cells entering at the cool
(leading) edge minus cells leaving at the warm (trailing) edge. They disagree in
sign when the leading edge still recruits farmable land but no longer recruits
*feasible* land, which happens once the window's displacement exceeds the headroom
between its leading edge and `S`'s upper bound along the migration axis.

Writing `Gamma` for the lapse rate, the displacement is `dz = dT / Gamma`, and the
condition for inversion is `dz > h`, with `h` the headroom. This is the part a
reader in another system can evaluate against their own landscape.

In [4]:
# Edge accounting: what enters at the cool edge vs what leaves at the warm edge
EPS = 0.05
print('cells entering/leaving per +/-0.05 C of band edge\n')
print(f"{'unit':>8} {'dT':>6} {'inC':>6} {'outC':>6} {'netC':>7} | {'inF':>6} {'outF':>6} {'netF':>7}")
for r, like in (('kona', s['kona_like']), ('kau', s['kau_like'])):
    mu, sg = P.envelope(FTemp, reg, idx, r); hw = P.HW*sg
    for dt in (0.0, P.DT_HORIZON['2045']):
        e = {}
        for nm, m in (('C', isl['farmable']), ('F', isl['farmable'] & like)):
            t = isl['T'][m]
            e[nm] = (int(((t >= mu-hw-dt-EPS) & (t < mu-hw-dt+EPS)).sum()),
                     int(((t >= mu+hw-dt-EPS) & (t < mu+hw-dt+EPS)).sum()))
        print(f'{r:>8} {dt:+6.2f} {e["C"][0]:6,d} {e["C"][1]:6,d} {e["C"][0]-e["C"][1]:+7,d} | '
              f'{e["F"][0]:6,d} {e["F"][1]:6,d} {e["F"][0]-e["F"][1]:+7,d}')

# Headroom: how far the leading edge travels before it clears S's ceiling
z = isl['X'][:, 0]
belt = isl['farmable'] & (s['kona_like'] | s['kau_like']) & ~np.isnan(isl['T'])
fit = np.polyfit(z[belt], isl['T'][belt], 1)
GAMMA = abs(fit[0])
DZ = P.DT_HORIZON['2045'] / GAMMA
matched['lapse_C_per_km'] = float(GAMMA*1000)
matched['displacement_m_2045'] = float(DZ)
print(f'\nbelt lapse {GAMMA*1000:.2f} C/km -> +{P.DT_HORIZON["2045"]} C displaces the window {DZ:.0f} m upslope\n')
for r, like in (('kona', s['kona_like']), ('kau', s['kau_like'])):
    mu, sg = P.envelope(FTemp, reg, idx, r); hw = P.HW*sg
    zS = z[isl['farmable'] & like]; zmax = np.percentile(zS, 97.5)
    zlead0 = (mu - hw - fit[1]) / fit[0]      # leading edge at dT=0
    h = zmax - zlead0
    matched[f'headroom_m_{r}'] = float(h)
    matched[f'displacement_over_headroom_{r}'] = float(DZ/h)
    print(f'{r}: S top {zmax:.0f} m, leading edge at dT=0 {zlead0:.0f} m, headroom {h:.0f} m')
    print(f'    dz/h = {DZ/h:.2f}  ->  {"INVERTS" if DZ > h else "no inversion"}')
    for dt in (0.0, P.DT_HORIZON['2035'], P.DT_HORIZON['2045']):
        zl = (mu - hw - dt - fit[1]) / fit[0]
        frac = 100*(zS > zl).sum()/len(zS)
        matched[f'S_above_lead_{r}_{dt:.2f}'] = float(frac)
        print(f'    dT={dt:+.2f}: leading edge {zl:.0f} m, {frac:.1f}% of S still above it')

json.dump(matched, open('data/matched.json','w'), indent=1)
print('\nwrote data/matched.json')


cells entering/leaving per +/-0.05 C of band edge

    unit     dT    inC   outC    netC |    inF   outF    netF
    kona  +0.00    279    310     -31 |     38     57     -19
    kona  +1.35    260    303     -43 |     16     56     -40
     kau  +0.00    315    300     +15 |     83     68     +15
     kau  +1.35    262    293     -31 |     18     80     -62

belt lapse 5.73 C/km -> +1.35 C displaces the window 236 m upslope

kona: S top 975 m, leading edge at dT=0 778 m, headroom 197 m
    dz/h = 1.19  ->  INVERTS
    dT=+0.00: leading edge 778 m, 17.6% of S still above it
    dT=+1.00: leading edge 952 m, 3.7% of S still above it
    dT=+1.35: leading edge 1013 m, 1.0% of S still above it
kau: S top 964 m, leading edge at dT=0 740 m, headroom 224 m
    dz/h = 1.05  ->  INVERTS
    dT=+0.00: leading edge 740 m, 23.4% of S still above it
    dT=+1.00: leading edge 915 m, 5.4% of S still above it
    dT=+1.35: leading edge 976 m, 2.1% of S still above it

wrote data/matched.json


## |F| is single-peaked, and the peak is at present

Sweeping `dT` in both directions. If the intersection simply "contracts under warming" it should fall monotonically; instead it rises to a maximum and falls away on both sides, so *displacement* is what costs, in whichever direction.

In [5]:
sweep = np.arange(-3.0, 3.01, 0.05)
curves = {}
for r, like in (('kona', s['kona_like']), ('kau', s['kau_like'])):
    mu, sg = P.envelope(FTemp, reg, idx, r)
    mask = isl['farmable'] & like
    curves[r] = np.array([P.feasible_size(isl['T'], mask, mu, sg, d) for d in sweep])
peak = {r: sweep[int(np.argmax(c))] for r, c in curves.items()}
show = [-1.5,-1.0,-0.5,0.0,0.5,1.0,1.35]
print(f"{'dT':>6} " + ' '.join(f'{r:>9}' for r in curves))
for d in show:
    i = int(np.argmin(np.abs(sweep-d)))
    print(f'{d:+6.2f} ' + ' '.join(f'{curves[r][i]:9,d}' for r in curves))
print(f"\npeak: " + ', '.join(f'{r} {peak[r]:+.2f} C' for r in peak))
ep = json.load(open('data/baseline_epoch.json'))
print(f"re-centred on end of record (shift {ep['offset_to_endrec']:+.2f} C): " +
      ', '.join(f"{r} {peak[r]-ep['offset_to_endrec']:+.2f} C" for r in peak))
print('\nBoth peaks sit at or below present climate: neither district is still')
print('approaching an optimum, and no warming pathway increases the feasible set.')

    dT      kona       kau
 -1.50     1,418     1,142
 -1.00     1,647     1,514
 -0.50     1,722     1,740
 +0.00     1,651     1,836
 +0.50     1,510     1,821
 +1.00     1,349     1,660
 +1.35     1,215     1,502

peak: kona -0.65 C, kau +0.15 C
re-centred on end of record (shift +0.49 C): kona -1.14 C, kau -0.34 C

Both peaks sit at or below present climate: neither district is still
approaching an optimum, and no warming pathway increases the feasible set.


## Pooled union — the analysis unit

Notebook 02 established that no district contrast is resolvable. So the reported quantity is the union: total feasible coffee ground on the island, which also pools the spatial blocks and buys real precision.

In [6]:
n0, _ = P.pooled_union(isl, FT, FTemp, reg, dt=0.0)
out = {'pooled_baseline_cells': n0}
for h, dt in P.DT_HORIZON.items():
    n, _ = P.pooled_union(isl, FT, FTemp, reg, dt=dt)
    out[f'pooled_decline_{h}'] = 100*(n-n0)/n0
    print(f'{h}: {n:,} cells, {100*(n-n0)/n0:+.1f}% from baseline {n0:,}')
json.dump(out, open('data/feasible_set.json','w'), indent=1)
reg_pn = json.load(open('../paper_numbers.json'))
print(f"\nregression check vs registered value: "
      f"{out['pooled_decline_2045']:+.1f}% vs {reg_pn['pooled_decline_2045']:+.1f}%")
assert abs(out['pooled_decline_2045'] - reg_pn['pooled_decline_2045']) < 0.5
print('matches')

2035: 3,009 cells, -13.7% from baseline 3,487
2045: 2,717 cells, -22.1% from baseline 3,487

regression check vs registered value: -22.1% vs -22.1%
matches


## Figure — the feasible set is single-peaked, and the peak is behind us

Grey is the thermal set alone; rust is the intersection with feasible ground. The intersection rises to a maximum and falls away on **both** sides, so displacement is what costs — not warming specifically. The dashed line marks the peak; the shaded band is the 2035–2045 horizon.

In [7]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os
os.makedirs('figures', exist_ok=True)
KONA, KAU, F_COL, C_COL = '#c1440e', '#1f6f8b', '#c1440e', '#888888'
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})

ep = json.load(open('data/baseline_epoch.json'))
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharex=True)
for ax, r, nm in zip(axes, ('kona','kau'), ('Kona', "Ka'u")):
    mu, sg = P.envelope(FTemp, reg, idx, r)
    # |C| in its OWN units on a twin axis -- no rescaling. |C| counts all farmable
    # island cells in the thermal band; |F| counts only those on feasible ground,
    # so the two live on genuinely different scales and must not share one.
    conly = np.array([P.feasible_size(isl['T'], isl['farmable'], mu, sg, d) for d in sweep])
    a2 = ax.twinx()
    a2.plot(sweep, conly, color=C_COL, lw=1.5)
    a2.set_ylim(0, conly.max()*1.25)
    a2.spines['top'].set_visible(False)
    a2.tick_params(axis='y', colors=C_COL)
    if r == 'kau':
        a2.set_ylabel(r'$|\mathcal{C}|$  farmable cells in band', color=C_COL)

    ax.plot(sweep, curves[r], color=F_COL, lw=2.4, zorder=3)
    ax.set_ylim(0, curves[r].max()*1.25)
    ax.set_zorder(a2.get_zorder()+1); ax.patch.set_visible(False)
    ax.axvline(0, color='k', ls=':', lw=1)
    ax.axvspan(1.00, 1.35, color=F_COL, alpha=.10)
    ax.axvline(peak[r], color=F_COL, ls='--', lw=1.2)
    ax.annotate(f'peak {peak[r]:+.2f} °C', xy=(0.02, 0.94), xycoords='axes fraction',
                fontsize=9, color=F_COL)
    ax.set_xlabel(r'$\Delta T$ (°C)'); ax.set_title(nm)
axes[0].set_ylabel(r'$|\mathcal{F}|=|\mathcal{C}\cap\mathcal{S}|$  feasible cells', color=F_COL)
axes[0].tick_params(axis='y', colors=F_COL)
from matplotlib.lines import Line2D
axes[0].legend(handles=[Line2D([],[],color=C_COL,lw=1.5,label=r'$|\mathcal{C}|$ climate only (right axis)'),
                        Line2D([],[],color=F_COL,lw=2.4,label=r'$|\mathcal{F}|$ screened (left axis)')],
               frameon=False, fontsize=8, loc='lower left')
fig.tight_layout(); fig.savefig('figures/04_F_vs_dT.png', dpi=200, bbox_inches='tight')
plt.close(fig); print('figures/04_F_vs_dT.png  (twin axis, real units)')

figures/04_F_vs_dT.png  (twin axis, real units)


## Figure — the inversion

The result the paper turns on. Climate alone says most of the island's farmable land gains suitability. Screened by ground coffee can actually occupy, the feasible set shrinks and the residual gaining fraction is a rounding error.

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.9))
ax = axes[0]
lbl = ['climate only\n(% farmable land\ngaining)', 'inside terrain\nfootprint\n(% still gaining)']
xs = np.arange(2); w = 0.36
for i,(r,col) in enumerate((('kona',KONA),('kau',KAU))):
    row = [x for x in rows if x[0]==r][0]
    ax.bar(xs + (i-0.5)*w, [row[2], row[3]], width=w, color=col, alpha=.8,
           label={'kona':'Kona','kau':"Ka'u"}[r])
    for j,v in enumerate([row[2], row[3]]):
        ax.text(xs[j]+(i-0.5)*w, v+1.2, f'{v:.1f}%', ha='center', fontsize=9)
ax.set_xticks(xs); ax.set_xticklabels(lbl, fontsize=9); ax.set_ylabel('% of farmable island land')
ax.set_ylim(0, 82); ax.legend(frameon=False); ax.set_title('Same warming, two answers')

ax = axes[1]
grid = np.full(isl['land'].shape, np.nan)
flatpos = np.where(isl['land'].ravel())[0]
code = np.zeros(len(isl['X']))
_, u_now = P.pooled_union(isl, FT, FTemp, reg, dt=0.0)
_, u_45  = P.pooled_union(isl, FT, FTemp, reg, dt=1.35)
code[isl['farmable']] = 0.5
code[u_now] = 1.5
code[u_now & ~u_45] = 2.5
grid.ravel()[flatpos] = code
# crop to the data extent -- the full raster is mostly ocean
rr, cc = np.where(isl['land'])
r0, r1, c0, c1 = rr.min()-3, rr.max()+4, cc.min()-3, cc.max()+4
grid = grid[max(0,r0):r1, max(0,c0):c1]
from matplotlib.colors import ListedColormap, BoundaryNorm
cm = ListedColormap(['#e8e8e8', '#7fb069', '#c1440e'])
ax.imshow(grid, cmap=cm, norm=BoundaryNorm([0,1,2,3], 3), interpolation='nearest')
ax.set_xticks([]); ax.set_yticks([])
ax.set_title(f'Feasible ground: retained (green) vs lost by 2045 (rust)')
import matplotlib.patches as mp
ax.legend(handles=[mp.Patch(color='#7fb069', label='feasible in 2045'),
                   mp.Patch(color='#c1440e', label='lost by 2045'),
                   mp.Patch(color='#e8e8e8', label='farmable, not feasible')],
          frameon=False, fontsize=8, loc='upper left', bbox_to_anchor=(0.0,-0.02))
fig.tight_layout(); fig.savefig('figures/04_inversion.png', dpi=200, bbox_inches='tight')
plt.close(fig); print('figures/04_inversion.png')

figures/04_inversion.png
